# Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import ToTensor, Compose, Normalize, Resize
from torchvision.datasets import MNIST
from torch.utils.data import random_split, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Setup Device

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(device)

# Load Data and Config

In [ ]:
train_data = MNIST(
    root="data",
    train=True,
    transform=Compose([ToTensor(), Normalize(mean=(0.1307,), std=(0.3081,))]),
    download=True,
)
test_data = MNIST(
    root="data",
    train=False,
    transform=Compose([ToTensor(), Normalize(mean=(0.1307,), std=(0.3081,))]),
    download=True,
)

loaders = {
    "train_data": DataLoader(
        train_data,
        batch_size=32,
        shuffle=True,
        num_workers=5,
    ),
    "test_data": DataLoader(
        test_data,
        batch_size=32,
        shuffle=True,
        num_workers=5,
    ),
}

# Architecture for MNIST Classification

In [ ]:
class MnistClassification(nn.Module):
    def __init__(self):
        super(MnistClassification, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)

        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)

        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Setup Optimizers and Loss Function

In [ ]:
model = MnistClassification().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.LinearLR(optimizer)
loss_fn = nn.CrossEntropyLoss()

# Define Train and Test Functions

In [ ]:
def train(epoch):
    model.train()
    train_loss = 0
    correct = 0
    n_train = len(loaders["train_data"].dataset)

    for batch_idx, (data, target) in enumerate(loaders["train_data"]):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
    train_loss /= n_train
    train_acc = 100 * correct / n_train
    print(
        f"Epoch: {epoch} - Train loss: {train_loss:.4f}, Train accuracy: {correct}/{n_train} ({train_acc:.4f}%)"
    )


def test():
    model.eval()
    test_loss = 0
    correct = 0
    n_test = len(loaders["test_data"].dataset)

    with torch.no_grad():
        for data, target in loaders["test_data"]:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += loss_fn(output, target).item() * data.size(0)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= n_test
    test_acc = 100 * correct / n_test
    print(
        f" Test loss: {test_loss:.4f}, Test accuracy: {correct}/{n_test} ({test_acc:.4f}%)\n"
    )

# Train and Test

In [ ]:
for epoch in range(10):
    train(epoch)
    test()

# Visualize Examples

In [ ]:
def infer(img_path: str, target: int) -> None:
    path = Path(img_path)
    model.eval()

    transform = Compose(
        [Resize((28, 28)), ToTensor(), Normalize(mean=(0.1307,), std=(0.3081,))]
    )
    if not path.exists():
        print(f" Error: file not found, the path doesn't exist: {img_path}")
        return

    if not path.is_file():
        print(f"Error: path is not a file: {img_path}")
        return 

    try:
        image = Image.open(img_path).convert("L")
    except Exception as e:
        print(f"error: {e}")
        return

    data = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(data)
    prediction = output.argmax(dim=1, keepdim=True).item()

    print(f"Prediction: {prediction}")
    print(f"Actual (target): {target}")
    print(f"Correct: {prediction == target}")

    plt.imshow(data.squeeze().cpu().numpy(), cmap="gray")
    plt.axis("off")
    plt.show()

In [ ]:
infer("example_img/a.png", 6)